# Analyzing Asylum Research AFM `.ibw` files with `igor2`This tutorial walks through a simple, end-to-end workflow for opening, inspecting, and plotting Atomic Force Microscope (AFM) images stored in **Igor Binary Wave** (`.ibw`) files using the [`igor2`](https://pypi.org/project/igor2/) Python package. It is written for people who are comfortable with AFM data but new to Python.

## What you will need- A working Python 3 environment. The commands here assume you are running inside a Jupyter notebook or a Python-friendly environment such as VS Code.- An `.ibw` file exported from Asylum Research AFM software.- Basic familiarity with your file system (where the data file is stored) is helpful, but no prior Python knowledge is required.

## 1. Install required packagesThe two packages we rely on are:- **igor2** for reading `.ibw` files.- **matplotlib** for plotting images.Run the cell below to install them. The `-q` flag keeps the output concise.

In [ ]:
# Install the igor2 reader and plotting library.# Remove "-q" if you want to see full installation logs.!pip install -q igor2 matplotlib

## 2. Import librariesEach import pulls common, beginner-friendly tools:- `os` is used to build file paths that work on Windows, macOS, and Linux.- `igor` comes from the `igor2` package and reads `.ibw` files.- `matplotlib.pyplot` (aliased as `plt`) plots images.- `pprint` is handy for printing nested dictionaries in a readable way.

In [ ]:
import osfrom pprint import pprintimport igorfrom matplotlib import pyplot as plt

## 3. Confirm the installation`help(igor)` prints the documentation that ships with the package. If the package was installed correctly, you should see an overview of the available modules and functions.

In [ ]:
help(igor)

## 4. Locate your `.ibw` fileUpdate the two variables below to point to your data:- `data_dir` – the folder containing your `.ibw` files.- `filename` – the name of the file you want to open.If your file is in the same directory as this notebook, you can leave `data_dir` empty (`""`). Otherwise, paste the full path to the folder. The code builds a combined `path` that works cross-platform.

In [ ]:
# TODO: Replace these with your own paths.data_dir = ""  # Example: r"C:\Users\you\Documents\afm" or "/home/you/data"filename = "test.ibw"  # Example: "my_afm_scan.ibw"path = os.path.join(data_dir, filename)print(f"Loading: {path}")

## 5. Load the `.ibw` file`igor.binarywave.load()` returns a dictionary containing both the raw data array and metadata. If the path is wrong, you will see a clear error message, so double-check the folder and file name if you run into trouble.

In [ ]:
file = igor.binarywave.load(path)print("Top-level keys:", file.keys())print("Wave keys:", file['wave'].keys())

## 6. Decode channel labelsAFM images typically store multiple channels (e.g., height, amplitude, phase) inside the same cube of data. Labels are stored as raw bytes, so we decode them to human-readable text and collect them in a simple Python list.

In [ ]:
labels = []for label in file['wave']['labels'][2][1:]:  # Skip the first placeholder entry.    labels.append(label.decode())print("Available channels:")for i, label in enumerate(labels):    print(f"  {i}: {label}")

## 7. Parse the measurement noteThe `note` field contains instrument settings and experimental notes as a single byte string. We split it into a dictionary for easier inspection. Certain degree and micro symbols are normalized so they render correctly in UTF-8.

In [ ]:
note = {}for s in file['wave']['note']        .replace(b'°', b'Â°')        .replace(b'µ', b'Âµ')        .decode()        .split(''):    s = s.strip()    if not s or ':' not in s:        continue    key, value = s.split(':', 1)    note[key.strip()] = value.strip()print("Parsed note entries:")pprint(note)

## 8. Extract the data array`file['wave']['wData']` is a NumPy array with the shape `(y_pixels, x_pixels, channels)`. Confirm the shape so you know how many channels you can visualize.

In [ ]:
data = file['wave']['wData']print("Data shape (rows, columns, channels):", data.shape)

## 9. Plot an AFM channelPick the channel you want to visualize—`HeightTrace` is a common choice. The snippet below locates the channel index by name, then plots it using a grayscale colormap. Feel free to experiment with other channel names from the `labels` list.

In [ ]:
channel_name = 'HeightTrace'  # Change this to any label printed above.try:    channel_index = labels.index(channel_name)except ValueError:    raise ValueError(f"Channel '{channel_name}' not found. Available: {labels}")plt.figure(figsize=(6, 6))plt.imshow(data[:, :, channel_index], cmap='gray')plt.title(f"AFM image: {channel_name}")plt.colorbar(label='Arbitrary units')plt.axis('off')plt.show()

## 10. Next steps- Try additional channels (e.g., `PhaseTrace`, `AmplitudeTrace`) by changing `channel_name`.- Use `plt.subplots` to compare multiple channels side-by-side.- Save figures with `plt.savefig("output.png", dpi=300)` for publication-ready images.- Explore other metadata stored under `file['wave']` to document your acquisition settings.